# 说明

代码使用了Azure AI Search服务，需要在Azure平台注册和配置，这里没跑，可以用参考CHROMADB改写，有兴趣可以自己试下
Azure AI Search是一个企业级搜索服务，它在原有的全文检索和语义排名能力的基础上，完全集成了高性能向量数据库的功能，旨在成为构建 RAG 架构和 Copilot 应用的最佳混合信息检索平台

这个 Jupyter Notebook 演示了如何构建一个具有持久记忆功能的 AI 旅行预订代理。它解决了 AI 代理的一个关键问题：如何记住用户偏好并在后续对话中使用这些信息。

具体来说，这个文件展示了：
1. 跨会话记忆能力：当用户 Sarah 第一次预订周年旅行时表达了对"浪漫酒店、水疗服务、无障碍设施"的偏好，几周后她再次预订家庭旅行时，代理能自动记住并应用这些偏好
2. 语义记忆检索：不仅能记住关键词，还能理解"饮食限制"与"素食"、"坚果过敏"等概念的关联
3. 个性化推荐：基于历史交互提供更精准的酒店推荐

## 实现步骤详解
### 1️⃣ 核心技术栈
- **Mem0**：作为智能记忆层，存储和检索用户偏好
- **Azure AI Search**：作为向量数据库，存储记忆和酒店数据
- **Semantic Kernel**：构建 AI 代理和插件系统

### 2️⃣ 实现流程
1. **初始化基础设施**
    - 配置 Azure AI Search 索引存储酒店数据
    - 设置 Mem0 使用 Azure AI Search 作为记忆存储
2. **创建记忆增强型插件**
    - 开发 `TravelBookingPlugin` 插件，包含：
        - `search_hotels()`：搜索酒店
        - `store_user_preference()`：存储用户偏好
        - `get_user_preferences()`：获取所有偏好
        - `search_memories()`：语义搜索记忆
3. **构建智能代理**
    - 创建 Semantic Kernel 代理，集成插件
    - 设置系统提示词指导代理如何使用记忆
4. **演示记忆工作流程**
    - 首次对话：存储用户偏好
    - 后续对话：自动检索并应用记忆
    - 语义搜索：理解"饮食限制"等概念

# 使用 Mem0、Semantic Kernel 和 Azure AI Search 构建具有持久记忆的 AI 代理

本笔记本演示如何构建一个智能旅行预订代理，该代理能够在对话中记住用户偏好。通过结合 Mem0、Semantic Kernel 和 Azure AI Search，我们创建了一个基于历史交互提供个性化旅行推荐的代理。

## 您将学习：
1. **Mem0 集成**：如何将 Mem0 用作 AI 代理的记忆层
2. **Azure AI Search 作为向量存储**：使用语义搜索存储和检索记忆
3. **持久用户偏好**：在不同聊天会话中记住用户偏好
4. **Semantic Kernel 插件**：构建同时利用记忆和搜索功能的插件

## 前提条件：
- 已配置 Azure OpenAI 部署
- 已创建 Azure AI Search 服务
- 了解基本的 Semantic Kernel 概念


## 理解内存架构

### 什么是 Mem0？

**Mem0** 是一个智能内存层，提供以下功能：
- **长期记忆**：存储用户偏好、过去的互动以及学习到的信息
- **语义搜索**：根据上下文检索相关记忆
- **用户特定存储**：为不同用户维护独立的内存空间
- **自动相关性**：根据当前上下文呈现最相关的记忆

### 各组件如何协同工作：
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│ Semantic        │────▶│      Mem0        │────▶│ Azure AI Search │
│ Kernel 智能体    │     │     记忆管理层    │     │    向量检索层    │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                       │                         │
         │                       │                         │
      处理用户请求          存储与检索用户偏好           保存记忆向量
                           及上下文信息                和旅行知识数据
```


In [ ]:
! pip install mem0ai

## 导入所需的包


In [2]:
import json
import os
from typing import Annotated, List, Dict, Any
from datetime import datetime
import uuid

from IPython.display import display, HTML, Markdown
from dotenv import load_dotenv

# Azure AI Search
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SearchField,
    VectorSearchAlgorithmMetric
)

# Mem0
from mem0 import Memory

# Semantic Kernel
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.functions import kernel_function
from semantic_kernel.contents import ChatHistory
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread

## 环境配置


In [ ]:
# Load environment variables
load_dotenv()

# Azure OpenAI Configuration
azure_openai_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")  # Use a recent API version


# Azure AI Search Configuration
search_service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY")

# Index names
travel_index_name = "travel-hotels"
memory_index_name = "mem0-memories"



## 初始化 Azure AI Search 以处理旅游数据

首先，我们将使用示例酒店和目的地数据设置 Azure AI Search，供我们的代理进行搜索。


In [ ]:
# Initialize search clients
# 创建索引客户端
index_client = SearchIndexClient(
    endpoint=search_service_endpoint,
    credential=AzureKeyCredential(search_api_key)
)

# Create travel data index if it doesn't exist
# 定义酒店数据的结构（就像设计数据库表）
travel_fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="name", type=SearchFieldDataType.String),
    SearchableField(name="description", type=SearchFieldDataType.String),
    SearchableField(name="location", type=SearchFieldDataType.String),
    SearchableField(name="amenities", type=SearchFieldDataType.String),
    SimpleField(name="price_per_night", type=SearchFieldDataType.Double),
    SimpleField(name="rating", type=SearchFieldDataType.Double),
    SearchableField(name="tags", type=SearchFieldDataType.String, collection=True)
]

# 创建酒店索引（如果不存在）
travel_index = SearchIndex(name=travel_index_name, fields=travel_fields)

try:
    index_client.get_index(travel_index_name)
    print(f"✅ 索引 '{travel_index_name}' 已存在")
except:
    index_client.create_index(travel_index)
    print(f"✅ 已创建索引 '{travel_index_name}'")


# Initialize search client for travel data
# 初始化搜索客户端（用于实际查询）
travel_search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=travel_index_name,
    credential=AzureKeyCredential(search_api_key)
)

In [ ]:
# Add sample travel data
# 添加示例酒店数据
sample_hotels = [
    {
        "id": "1",
        "name": "Le Meurice Paris",
        "description": "Luxury palace hotel with Michelin-starred dining and views of the Tuileries Garden",
        "location": "Paris, France",
        "amenities": "Spa, Michelin Restaurant, Concierge, Room Service, Fitness Center",
        "price_per_night": 850,
        "rating": 4.8,
        "tags": ["luxury", "romantic", "historic", "fine-dining", "spa"]
    },
    {
        "id": "2",
        "name": "Four Seasons Maui",
        "description": "Beachfront resort with world-class spa and family-friendly activities",
        "location": "Maui, Hawaii",
        "amenities": "Beach Access, Kids Club, Multiple Pools, Spa, Golf Course",
        "price_per_night": 695,
        "rating": 4.7,
        "tags": ["beach", "family-friendly", "resort", "spa", "golf"]
    },
    {
        "id": "3",
        "name": "Aman Tokyo",
        "description": "Minimalist luxury hotel with panoramic city views and traditional onsen",
        "location": "Tokyo, Japan",
        "amenities": "Onsen, City Views, Fine Dining, Spa, Business Center",
        "price_per_night": 780,
        "rating": 4.9,
        "tags": ["luxury", "business", "spa", "city", "minimalist"]
    },
    {
        "id": "4",
        "name": "Hotel Sacher Vienna",
        "description": "Historic hotel home of the original Sachertorte with elegant rooms",
        "location": "Vienna, Austria",
        "amenities": "Historic Cafe, Concierge, Accessible Rooms, Pet-Friendly",
        "price_per_night": 420,
        "rating": 4.6,
        "tags": ["historic", "accessible", "pet-friendly", "cultural", "cafe"]
    },
    {
        "id": "5",
        "name": "Fairmont Whistler",
        "description": "Ski-in/ski-out resort with family suites and mountain views",
        "location": "Whistler, Canada",
        "amenities": "Ski Access, Family Suites, Heated Pool, Kids Programs",
        "price_per_night": 380,
        "rating": 4.5,
        "tags": ["ski", "family-friendly", "mountain", "resort", "accessible"]
    }
]

# Upload hotels to search index
travel_search_client.upload_documents(documents=sample_hotels)
print(f"✅ Uploaded {len(sample_hotels)} hotels to search index")
print(f"✅ 已上传 {len(sample_hotels)} 家酒店到搜索索引")

## 使用 Azure AI Search 初始化 Mem0

现在我们将配置 Mem0 使用 Azure AI Search 作为其持久内存的向量存储。


In [ ]:
# Mem0配置 - 告诉它如何与Azure AI Search通信
mem0_config = {
    "llm": {
        "provider": "azure_openai",
        "config": {
            "model": azure_openai_deployment,
            "temperature": 0.2,  # 控制AI创造力（0=最确定，1=最创意）
            "max_tokens": 1500,  # 回复的最大长度
            "azure_kwargs": {
                "azure_deployment": azure_openai_deployment,
                "api_version": api_version,
                "azure_endpoint": azure_openai_endpoint,
                "api_key": azure_openai_api_key,
            }
        }
    },
    "vector_store": {
        "provider": "azure_ai_search",
        "config": {
            # 从endpoint中提取服务名称（格式：https://<service-name>.search.windows.net）
            "service_name": search_service_endpoint.split("//")[1].split(".")[0],
            "api_key": search_api_key,
            "collection_name": "mem0",  # 在Azure Search中的索引名称
            "embedding_model_dims": 1536  # 使用的嵌入模型维度
        }
    },
    "embedder": {
        "provider": "azure_openai",
        "config": {
            "model": "text-embedding-ada-002",  # 用于生成文本向量的模型
            "azure_kwargs": {
                "azure_deployment": "text-embedding-ada-002",  # Update if different
                "api_version": api_version,
                "azure_endpoint": azure_openai_endpoint,
                "api_key": azure_openai_api_key,
            }
        }
    }
}

# Initialize Mem0
# 初始化Mem0记忆系统
memory = Memory.from_config(mem0_config)
print("🧪 测试Mem0设置...")
# 测试记忆存储
test_messages = [
    {"role": "user", "content": "I prefer luxury hotels with spa services."},
    {"role": "assistant", "content": "I'll remember you prefer luxury hotels with spa services for future recommendations."}
]
# 添加测试记忆（使用测试用户ID）
memory.add(test_messages, user_id="test_user", metadata={"category": "preferences"})
# 检查是否成功存储
test_memories = memory.get_all(user_id="test_user")
print(f"✅ Mem0测试成功！找到 {len(test_memories)} 条记忆")


## 创建旅行预订插件

此插件通过 Mem0 提供搜索酒店和管理用户偏好的功能。


In [ ]:
class TravelBookingPlugin:
    """Plugin for searching hotels and managing user travel preferences
    这个插件让AI代理能够搜索酒店和管理用户偏好"""

    def __init__(self, search_client: SearchClient, memory: Memory):
        self.search_client = search_client
        self.memory = memory

    @kernel_function(
        description="Search for hotels based on criteria like location, amenities, or tags"
    )
    def search_hotels(
        self,
        query: Annotated[str, "Search query for hotels (location, amenities, etc.)"],
        max_results: Annotated[int, "Maximum number of results to return"] = 3
    ) -> Annotated[str, "List of hotels matching the search criteria"]:
        """Search for hotels in the travel database
        根据位置、设施或标签等条件搜索酒店"""
        # 执行搜索查询
        results = self.search_client.search(
            search_text=query,
            top=max_results,
            include_total_count=True
        )

        # 整理搜索结果
        hotels = []
        for result in results:
            hotels.append({
                "name": result["name"],
                "location": result["location"],
                "description": result["description"],
                "price_per_night": result["price_per_night"],
                "rating": result["rating"],
                "amenities": result["amenities"],
                "tags": result["tags"]
            })

        return json.dumps(hotels, indent=2)

    @kernel_function(
        description="Store user travel preferences and important information in memory"
    )
    def store_user_preference(
        self,
        user_id: Annotated[str, "User identifier"],
        preference: Annotated[str, "User preference or information to remember"]
    ) -> Annotated[str, "Confirmation of stored preference"]:
        """Store user preferences in Mem0 memory
        将用户偏好存储到Mem0记忆中"""
        print(f"DEBUG: Storing preference for {user_id}: {preference}")

        try:
            # Simply add the preference to memory
            # 将偏好添加到记忆系统
            self.memory.add(preference, user_id=user_id)
            return f"✅ 已存储: {preference}"
        except Exception as e:
            return f"❌ 存储偏好时出错: {str(e)}"
        
    @kernel_function(
        description="Get all stored preferences for a user"
    )
    def get_user_preferences(
        self,
        user_id: Annotated[str, "User identifier"]
    ) -> Annotated[str, "All user preferences and memories"]:
        """Get all memories for a specific user
        获取特定用户的所有记忆"""
        print(f"DEBUG: Getting all preferences for {user_id}")

        try:
            # Get all memories for the user
            # 从记忆系统获取所有记忆
            results = self.memory.get_all(user_id=user_id)
            
            # 处理可能的响应格式
            # Handle the dict response with 'results' key
            if isinstance(results, dict) and 'results' in results:
                results = results.get('results', [])

            if not results:
                return f"未找到用户 {user_id} 的偏好"

            # Format results
            # 格式化记忆结果
            memories = []
            for result in results:
                if isinstance(result, dict):
                    memory_text = result.get('memory', str(result))
                    memories.append(memory_text)
                else:
                    memories.append(str(result))

            return f"User preferences for {user_id}:\n- " + "\n- ".join(memories)

        except Exception as e:
            print(f"获取偏好时出错: {str(e)}")
            return f"No preferences found for user {user_id}"


    @kernel_function(
        description="Search user's memories for relevant information"
    )
    def search_memories(
        self,
        user_id: Annotated[str, "User identifier"],  # 用户唯一标识
        query: Annotated[str, "What to search for (e.g., 'family vacation', 'dietary restrictions')"]  # 搜索关键词（如"家庭度假"）
    ) -> Annotated[str, "Relevant memories"]:
        """Search user memories using Mem0
        使用Mem0语义搜索用户记忆"""
        print(f"DEBUG: Searching memories for {user_id} with query: '{query}'")

        try:
            # Let Mem0 handle the search and ranking
            # 执行语义搜索
            results = self.memory.search(query, user_id=user_id)
            
            # 处理可能的响应格式
            # Handle the dict response with 'results' key
            if isinstance(results, dict) and 'results' in results:
                results = results.get('results', [])

            if not results:
                return f"No memories found for query: {query}"

            # Format results
            # 格式化搜索结果
            memories = []
            for result in results:
                if isinstance(result, dict):
                    memory_text = result.get('memory', str(result))
                    # Include relevance score if available
                    score = result.get('score', None)
                    if score:
                        memories.append(
                            f"{memory_text} (relevance: {score:.2f})")
                    else:
                        memories.append(memory_text)
                else:
                    memories.append(str(result))

            return "Relevant memories:\n- " + "\n- ".join(memories)

        except Exception as e:
            print(f"ERROR: {str(e)}")
            return "No memories found."
        
        
    

## 初始化语义内核代理

使用旅行预订插件创建我们的旅行预订代理。


In [ ]:
# Create the kernel
# 创建Semantic Kernel核心
kernel = Kernel()

# Add Azure OpenAI service
# 添加Azure OpenAI服务（AI的大脑）
chat_service = AzureChatCompletion(
    deployment_name=azure_openai_deployment,
    endpoint=azure_openai_endpoint,
    api_key=azure_openai_api_key,
)
kernel.add_service(chat_service)

# Create and add the travel booking plugin
# 创建旅行预订插件实例
travel_plugin = TravelBookingPlugin(travel_search_client, memory)
# 将插件添加到Kernel
kernel.add_plugin(
    plugin_name="TravelBooking",
    plugin=travel_plugin
)

# Create the travel agent
# 创建AI旅行代理
# 你是一个具有记忆功能的个性化旅行预订助手。

# 工作流程：
# 1. 当用户寻求帮助时，使用search_memories()搜索相关记忆
# 2. 使用记忆信息个性化回复
# 3. 存储用户提到的新偏好（使用store_user_preference()）
# 4. 当用户预订新旅行时，先检索用户的偏好（位置、设施、预算等）
# 5. 使用search_hotels()查找合适的酒店选项
# 6. 不要推荐超出预算的酒店

# 重要：对于所有记忆操作，必须使用user_id='sarah_johnson_123'
travel_agent = ChatCompletionAgent(
    service=chat_service,
    name="TravelBookingAssistant",
    instructions="""
    You are a personalized travel booking assistant with memory.
    
    WORKFLOW:
    1. When a user asks for help, search their memories using search_memories() with a relevant query
    2. Use the memories to personalize your response
    3. Store any new preferences they mention using store_user_preference()
    4. When the users is booking a new trip, first retrieve the users general travel preferences of the user by creating queries for hotels, dietary restrictions, location, amenities and budget. THEN use search_hotels() to find suitable options.
    5. Do not recommend hotels that are over budget. 
    
    IMPORTANT: For ALL memory operations (search_memories and store_user_preference), 
    you MUST use user_id='sarah_johnson_123' exactly as written.

    Example queries:
    - User asks about booking a trip → search_memories(query="preferences")
    - User asks about booking a trip → search_memories(query="dietary restrictions")
    - User asks about booking a trip → search_memories(query="location")
    - User asks about booking a trip → search_memories(query="amenities")
    - User asks about booking a trip → search_memories(query="budget")

    Always acknowledge what you found in their memories when responding.""",
    plugins=[travel_plugin]
)

## 用于清晰显示的辅助函数


In [ ]:
def display_message(role: str, content: str, color: str = "#2E8B57", emoji: str = ""):
    """Display a message with nice formatting"""
    html = f"""
    <div style='
        margin: 10px 0; 
        padding: 15px 20px; 
        border-left: 4px solid {color}; 
        background: rgba(128, 128, 128, 0.05); 
        border-radius: 8px;
    '>
        <strong style='color: {color}; font-size: 16px;'>{emoji} {role}:</strong><br>
        <div style='margin-top: 10px; white-space: pre-wrap; font-size: 14px; line-height: 1.6;'>{content}</div>
    </div>
    """
    display(HTML(html))

def display_memory_operation(operation: str, details: str, color: str = "#9370DB"):
    """Display memory operations for educational purposes"""
    html = f"""
    <div style='
        margin: 5px 20px;
        padding: 10px 15px;
        background: rgba(147, 112, 219, 0.1);
        border: 1px solid {color};
        border-radius: 6px;
        font-family: monospace;
        font-size: 13px;
    '>
        <strong style='color: {color};'>🧠 Memory {operation}:</strong>
        <div style='margin-top: 5px; color: #555;'>{details}</div>
    </div>
    """
    display(HTML(html))

def display_function_call(function_name: str, args: dict, result: str = None):
    """Display function calls for transparency"""
    html = f"""
    <details style='margin: 5px 20px; padding: 10px; background: rgba(0, 123, 255, 0.05); border: 1px solid #007BFF; border-radius: 6px;'>
        <summary style='cursor: pointer; font-weight: bold; color: #007BFF;'>⚙️ Function Call: {function_name}</summary>
        <div style='margin-top: 10px; font-family: monospace; font-size: 12px;'>
            <div><strong>Arguments:</strong> {json.dumps(args, indent=2)}</div>
    """
    if result:
        html += f"<div style='margin-top: 10px;'><strong>Result:</strong><pre style='background: #f8f8f8; padding: 8px; border-radius: 4px; overflow-x: auto;'>{result}</pre></div>"
    html += "</div></details>"
    display(HTML(html))

## 演示带记忆功能的旅行预订

让我们通过真实的旅行预订场景来展示代理如何记住并使用用户偏好。


### 场景 1：首次使用者 - 周年旅行计划


In [ ]:
# User ID for our demonstration
# 用户ID（实际应用中应从登录系统获取）
sarah_user_id = "sarah_johnson_123"

print("🎯 SCENARIO 1: Sarah's First Booking - Anniversary Trip\n")

# Create a new chat history for Sarah
# 创建聊天历史
sarah_chat = ChatHistoryAgentThread()

# First conversation
# 你好！我是Sarah，正在计划我们10周年的特别旅行。
# 我们喜欢浪漫的目的地、精致餐饮和水疗体验。
# 我丈夫有行动障碍，所以我们需要无障碍设施。
# 预算大约是每晚700-800美元。
user_message1 = """Hi! I'm Sarah and I'm planning a special trip for my 10th wedding anniversary. 
We love romantic destinations, fine dining, and spa experiences. My husband has mobility issues, 
so we need accessible accommodations. Our budget is around $700-800 per night."""

display_message("Sarah", user_message1, "#4fc3f7", "👤")

# Extract and display function calls for educational purposes
# Process with agent
response_content = ""
function_calls_made = []

async for response in travel_agent.invoke(
    messages=user_message1,
    thread=sarah_chat
):
    if response.content:
        response_content = str(response.content)

    # Parse function calls from the thread
    if hasattr(response, 'thread'):
        sarah_thread = response.thread

        # Check for function calls in the latest messages
        async for msg in sarah_thread.get_messages():
            if hasattr(msg, 'items') and msg.items:
                for item in msg.items:
                    if hasattr(item, 'function_invoke_attempt') and item.function_invoke_attempt:
                        func_call = item.function_invoke_attempt
                        function_info = {
                            'name': func_call.function_name,
                            'arguments': func_call.arguments,
                            'result': item.function_result.value if hasattr(item, 'function_result') else None
                        }
                        if function_info not in function_calls_made:
                            function_calls_made.append(function_info)

                            # Display the actual function calls
                            if 'get_user_preferences' in func_call.function_name:
                                display_memory_operation(
                                    "Retrieval", f"Checking existing preferences for user: {func_call.arguments.get('user_id', sarah_user_id)}")
                            elif 'store_user_preference' in func_call.function_name:
                                display_memory_operation(
                                    "Storage", f"Storing: {func_call.arguments.get('preference', '')}")
                            elif 'search_hotels' in func_call.function_name:
                                display_function_call(
                                    func_call.function_name,
                                    func_call.arguments,
                                    item.function_result.value if hasattr(
                                        item, 'function_result') else None
                                )

display_message("Travel Assistant", response_content, "#81c784", "🤖")

In [ ]:
# 萨赫酒店听起来很完美！我们俩都是素食主义者，而且我对坚果严重过敏。
# 您能详细介绍一下他们的餐饮选择吗？
user_message2 = """The Hotel Sacher sounds perfect! We're both vegetarian and I have a severe nut allergy. 
Can you tell me more about their dining options?"""

display_message("Sarah", user_message2, "#4fc3f7", "👤")

response2_content = ""
async for response in travel_agent.invoke(
    messages=user_message2,
    thread=sarah_thread
):
    if response.content:
        response2_content = str(response.content)

    if hasattr(response, 'thread'):
        sarah_thread = response.thread

        # Parse new function calls
        async for msg in sarah_thread.get_messages():
            if hasattr(msg, 'items') and msg.items:
                for item in msg.items:
                    if hasattr(item, 'function_invoke_attempt') and item.function_invoke_attempt:
                        func_call = item.function_invoke_attempt
                        if 'store_user_preference' in func_call.function_name and func_call.arguments.get('preference'):
                            pref = func_call.arguments.get('preference', '')
                            if 'vegetarian' in pref.lower() or 'nut allergy' in pref.lower():
                                display_memory_operation(
                                    "Storage", f"Storing: {pref}")

display_message("Travel Assistant", response2_content, "#81c784", "🤖")

In [ ]:

# After running all scenarios, verify memories are stored
from azure.search.documents import SearchClient
print("\n\n🔍 VERIFYING MEM0 STORAGE\n")

# Check Azure AI Search directly
mem0_search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name="mem0",
    credential=AzureKeyCredential(search_api_key)
)

try:
    # Count documents in the index
    results = mem0_search_client.search(
        search_text="*", include_total_count=True)
    total_docs = results.get_count()
    print(f"📊 Total documents in Mem0 index: {total_docs}")

    # Show first few documents
    print("\nSample documents:")
    for i, doc in enumerate(results):
        if i < 3:  # Show first 3
            print(f"\nDocument {i+1}:")
            print(f"  ID: {doc.get('id', 'N/A')}")
            print(f"  User ID: {doc.get('user_id', 'N/A')}")
            print(f"  Memory: {doc.get('payload', 'N/A')}")
except Exception as e:
    print(f"❌ Error checking Mem0 index: {str(e)}")
    print("The index might not be created yet or might be empty.")

In [ ]:
# Enhanced verification to debug Mem0 responses
print("🔍 ENHANCED MEM0 VERIFICATION\n")

# Create a unique test user
test_user = f"debug_user_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
test_memory = "I am vegetarian with a peanut allergy and love beach destinations"

print(f"1. Adding memory for {test_user}...")
add_result = memory.add(test_memory, user_id=test_user)
print(f"   Raw add result: {add_result}")

# Check if the result has a 'results' key
if isinstance(add_result, dict) and 'results' in add_result:
    actual_results = add_result.get('results', [])
    print(
        f"   Actual memories added: {len(actual_results) if isinstance(actual_results, list) else 0}")
    if isinstance(actual_results, list):
        for mem in actual_results:
            print(
                f"   - ID: {mem.get('id', 'N/A')}, Memory: {mem.get('memory', 'N/A')}")

print("\n2. Testing get_all()...")
all_mems = memory.get_all(user_id=test_user)
print(f"   Raw response: {all_mems}")
print(f"   Response type: {type(all_mems)}")

# Check if it's a dict with 'results' key
if isinstance(all_mems, dict):
    print(f"   Dict keys: {list(all_mems.keys())}")
    if 'results' in all_mems:
        results_value = all_mems['results']
        print(f"   'results' value type: {type(results_value)}")
        print(f"   'results' value: {results_value}")

        # If results is actually a list, show the memories
        if isinstance(results_value, list):
            print(f"   Number of memories: {len(results_value)}")
            for i, mem in enumerate(results_value):
                print(f"   Memory {i}: {mem}")

print("\n3. Testing search()...")
search_results = memory.search("peanut allergy", user_id=test_user)
print(f"   Raw response: {search_results}")
print(f"   Response type: {type(search_results)}")

# Check if it's a dict with 'results' key
if isinstance(search_results, dict):
    print(f"   Dict keys: {list(search_results.keys())}")
    if 'results' in search_results:
        results_value = search_results['results']
        print(f"   'results' value type: {type(results_value)}")
        print(f"   'results' value: {results_value}")

print("\n4. Testing direct API access...")
# Try to access memories through Azure AI Search directly
try:
    mem0_search_client = SearchClient(
        endpoint=search_service_endpoint,
        index_name="mem0",
        credential=AzureKeyCredential(search_api_key)
    )

    # Wait a moment for indexing
    import time
    time.sleep(2)

    # Search for the test user's memories
    azure_results = mem0_search_client.search(
        search_text="*",
        filter=f"user_id eq '{test_user}'",
        include_total_count=True
    )

    print(f"   Documents found in Azure: {azure_results.get_count()}")
    for doc in azure_results:
        payload = doc.get('payload', {})
        if isinstance(payload, str):
            import json
            try:
                payload = json.loads(payload)
            except:
                pass
        print(f"   - Memory: {payload}")

except Exception as e:
    print(f"   Error: {e}")

print("\n5. Testing Mem0 version...")
# Check if we need to use a different method or property
if hasattr(memory, '__version__'):
    print(f"   Mem0 version: {memory.__version__}")
if hasattr(memory, 'version'):
    print(f"   Mem0 version: {memory.version}")

# Check available methods
print("\n6. Available memory methods:")
for attr in dir(memory):
    if not attr.startswith('_') and callable(getattr(memory, attr)):
        print(f"   - {attr}")

### 情景 2：回访 - 家庭度假（数周后）


In [ ]:
print("\n\n🎯 SCENARIO 2: Sarah Returns Weeks Later for Family Vacation\n")
print("📅 Simulating time passing... Sarah starts a new conversation\n")

# Create a new thread to simulate a new conversation
sarah_thread_new = ChatHistoryAgentThread()

user_message3 = "Hi, my husband and I are planning another trip. We are looking for a good hotel!"

display_message("Sarah", user_message3, "#4fc3f7", "👤")

response3_content = ""
memories_retrieved = []

async for response in travel_agent.invoke(
    messages=user_message3,
    thread=sarah_thread_new
):
    if response.content:
        response3_content = str(response.content)

    if hasattr(response, 'thread'):
        sarah_thread_new = response.thread

        # Parse function calls
        async for msg in sarah_thread_new.get_messages():
            if hasattr(msg, 'items') and msg.items:
                for item in msg.items:
                    if hasattr(item, 'function_invoke_attempt') and item.function_invoke_attempt:
                        func_call = item.function_invoke_attempt

                        # Check for memory retrieval
                        if 'get_user_preferences' in func_call.function_name and hasattr(item, 'function_result'):
                            result = item.function_result.value
                            if result and "User preferences" in result:
                                display_memory_operation(
                                    "Retrieval", f"Found memories for {sarah_user_id}:\n{result}")

                        # Check for new preference storage
                        elif 'store_user_preference' in func_call.function_name:
                            display_memory_operation(
                                "Storage", f"Storing: {func_call.arguments.get('preference', '')}")

                        # Check for hotel search
                        elif 'search_hotels' in func_call.function_name:
                            display_function_call(
                                func_call.function_name,
                                func_call.arguments,
                                item.function_result.value if hasattr(
                                    item, 'function_result') else None
                            )

display_message("Travel Assistant", response3_content, "#81c784", "🤖")

In [ ]:
# Follow-up question
user_message4 = "Great suggestions! For the Maui option, what activities would you recommend for the kids?"

display_message("Sarah", user_message4, "#4fc3f7", "👤")

response4_content = ""
async for response in travel_agent.invoke(
    messages=user_message4,
    thread=sarah_thread_new
):
    if response.content:
        response4_content = str(response.content)
    sarah_thread_new = response.thread

display_message("Travel Assistant", response4_content, "#81c784", "🤖")

In [ ]:
print("\n🧪 TESTING MEMORY RETRIEVAL\n")

# First, ensure Sarah has some memories
test_preference = "I love romantic destinations with spa services"
result = travel_plugin.store_user_preference(sarah_user_id, test_preference)
print(f"Store result: {result}")

# Now test retrieval
preferences = travel_plugin.get_user_preferences(sarah_user_id)
print(f"\nRetrieved preferences:\n{preferences}")

# Also test the memory object directly
direct_memories = memory.get_all(user_id=sarah_user_id)
print(f"\nDirect memory.get_all() returned {len(direct_memories)} memories")
for i, mem in enumerate(direct_memories):
    print(f"Memory {i}: {mem}")

In [ ]:

# Check the Mem0 index structure in Azure AI Search
print("\n🔍 CHECKING MEM0 INDEX STRUCTURE\n")

try:
    # Get the mem0 index
    mem0_index = index_client.get_index("mem0")
    print("Mem0 index fields:")
    for field in mem0_index.fields:
        print(f"  - {field.name}: {field.type}")

    # Query the index directly
    mem0_search_client = SearchClient(
        endpoint=search_service_endpoint,
        index_name="mem0",
        credential=AzureKeyCredential(search_api_key)
    )

    # Get all documents for Sarah
    results = mem0_search_client.search(
        search_text="*",
        filter=f"user_id eq '{sarah_user_id}'",
        include_total_count=True
    )

    print(f"\nDocuments for {sarah_user_id}: {results.get_count()}")
    for doc in results:
        print(f"\nDocument ID: {doc.get('id')}")
        for key, value in doc.items():
            if key != 'id':
                print(
                    f"  {key}: {value[:100] if isinstance(value, str) and len(value) > 100 else value}")

except Exception as e:
    print(f"Error checking index: {str(e)}")

## 演示语义记忆搜索

Mem0的强大之处在于语义搜索——根据含义而不仅仅是关键词来查找相关记忆。


In [ ]:
print("🔍 SEMANTIC MEMORY SEARCH DEMONSTRATION\n")

# Search Sarah's memories for dietary-related information
dietary_search = memory.search(
    "dietary food allergies restrictions", user_id=sarah_user_id)

# Handle the dict response with 'results' key
if isinstance(dietary_search, dict) and 'results' in dietary_search:
    dietary_results = dietary_search.get('results', [])
else:
    dietary_results = dietary_search if isinstance(
        dietary_search, list) else []

print("Search Query: 'dietary food allergies restrictions'")
print(f"Results for Sarah:")
print("=" * 50)
if dietary_results:
    for mem in dietary_results:
        if isinstance(mem, dict):
            print(f"- {mem.get('memory', 'Unknown')}")
            print(f"  Relevance Score: {mem.get('score', 'N/A')}")
        else:
            print(f"- {mem}")
else:
    print("- No memories found")



## 关键要点

### 1. 持久的用户记忆
- **跨会话持久性**：用户偏好在不同对话中得以保留
- **用户隔离**：每个用户都有自己的记忆空间
- **自动上下文**：代理会自动检索相关记忆

### 2. Mem0 的优势
- **语义理解**：根据含义而非精确匹配来检索记忆
- **可扩展性**：使用 Azure AI Search 提供企业级存储
- **隐私**：用户记忆是隔离且安全的

### 3. 改进的用户体验
- **无需重复**：用户无需重复表达偏好
- **个性化**：推荐随着时间推移不断优化
- **上下文感知**：代理能够理解用户历史


## 摘要

恭喜你！你已经成功构建了一个具有持久记忆功能的AI旅行助手，使用了以下技术：

- **Mem0**：用于智能的持久记忆管理
- **Azure AI Search**：作为记忆和旅行数据的可扩展向量存储
- **Semantic Kernel**：用于协调代理和插件

## 你学到了什么：
1. 如何将 Mem0 与 Azure AI Search 集成以实现持久记忆
2. 构建利用记忆的 Semantic Kernel 插件
3. 创建能够跨会话记住用户偏好的代理
4. 使用语义搜索检索相关记忆

## 实际应用场景：
- **客户服务**：记住客户历史和偏好
- **个人助理**：在几天或几周内保持上下文
- **医疗保健**：跟踪患者信息和偏好
- **教育**：记住学生的学习进度和学习风格
- **电子商务**：基于历史记录进行个性化购物推荐

## 下一步：
- 实现记忆过期功能以处理时间敏感信息
- 添加记忆重要性评分
- 构建具有共享记忆的多代理系统
- 与CRM系统集成以支持企业应用
- 添加记忆版本控制和审计记录



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保准确性，但请注意，自动翻译可能包含错误或不准确之处。应以原始语言的文档为权威来源。对于关键信息，建议使用专业人工翻译。因使用本翻译而引起的任何误解或误读，我们概不负责。
